In [ ]:
import os
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader,Dataset
from torchvision import datasets,transforms
from PIL import Image
import numpy as np
from tqdm import tqdm


In [ ]:
import torch
print(torch.device)
print(type(torch.device))
print(torch.cuda.is_available())

<class 'torch.device'>
<class 'type'>
False


In [ ]:
from torchvision.transforms.transforms import Normalize
train_transform=transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((224,224)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(0.3),
    transforms.ToTensor(),
    # transforms.Normalize([0.5], [0.5])
    transforms.Lambda(lambda x: torch.clamp(x, -1000, 400)/1400)

])
test_transform=transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((224,224)),
    transforms.transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(0.3),
    transforms.transforms.ToTensor(),
    # transforms.Normalize([0.5],[0.5])
    transforms.Lambda(lambda x:torch.clamp(x,-1000,400)/1400)

])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
train_bath="/content/drive/MyDrive/big_data/deep/train"
test_path="/content/drive/MyDrive/big_data/deep/test"

In [ ]:
train_data=datasets.ImageFolder(root=train_bath,
                                transform=train_transform,
                                 target_transform=None)

test_data=datasets.ImageFolder(root=test_path,
                               transform=test_transform,
                               target_transform=None
                               )

In [ ]:
train_data.classes
test_data.classes

['normalBT', 'strokeBT']

In [ ]:
train_data_loader=DataLoader(dataset=train_data,
                             batch_size=32,
                             shuffle=True,
                             num_workers=2,
                             pin_memory=True
                             )

test_data_loader=DataLoader(dataset=test_data,
                            batch_size=32,
                            shuffle=False,
                            num_workers=2,
                            pin_memory=True
                            )

In [ ]:
class Simple_model(nn.Module):
  def __init__(self,input_shape,hidden_units,output_shape):
    super().__init__()
    self.block1=nn.Sequential(
        nn.Conv2d(in_channels=input_shape,out_channels=hidden_units,kernel_size=(4,4),
                  stride=1,padding=2),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units*2,kernel_size=(4,4),stride=1,padding=2),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
)

    self.block2=nn.Sequential(
        nn.Conv2d(in_channels=hidden_units*2,out_channels=hidden_units,kernel_size=(4,4),stride=1,padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units*2,out_channels=hidden_units*2,kernel_size=(4,4),stride=1,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)

    )

    self.classifier=nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_units*2*56*56,out_features=output_shape)
    )

  def forward(self,x:torch.tensor):
    x=self.block1(x)
    x=self.block2(x)
    x=self.classifier(x)
    return x




In [ ]:
from torch.nn.modules import pooling
class Simple_model(nn.Module):
  def __init__(self,input_shape,hidden_units,output_shape):
    super().__init__()
    self.block1=nn.Sequential(
        nn.Conv2d(in_channels=input_shape,out_channels=hidden_units,kernel_size=3,stride=1,padding=2),
        nn.ReLU(),
        nn.Conv2d(hidden_units,hidden_units*2,kernel_size=3,stride=1,padding=2),
        nn.MaxPool2d(kernel_size=2,stride=2)

    )


    self.block2=nn.Sequential(
        nn.Conv2d(in_channels=hidden_units*2,out_channels=hidden_units,kernel_size=3,stride=1,padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units*2,kernel_size=3,stride=1,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )

    self.last_conv=nn.Conv2d(in_channels=hidden_units*2,out_channels=output_shape,kernel_size=1)#why we have using that with our data
    self.pool=nn.AdaptiveAvgPool2d((1,1))





  def forward(self,x):
    x=self.block1(x)
    x=self.block2(x)
    x=self.last_conv(x)
    heatmap=x.clone()
    x=self.pool(x)
    x = x.view(x.size(0), -1)
    return x,heatmap

In [ ]:
class Modelv1(nn.Module):
  def __init__(self,input_shape,hidden_units,output_shape):
    super().__init__()
    self.block1=nn.Sequential(
        nn.Conv2d(in_channels=input_shape,out_channels=hidden_units,kernel_size=3,stride=1,padding=2),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        # nn.Dropout(0.2),
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units*2,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(hidden_units*2),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.MaxPool2d(kernel_size=2,stride=2),

    )
    self.block2=nn.Sequential(
        nn.Conv2d(in_channels=hidden_units*2,out_channels=hidden_units*2,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(hidden_units*2),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units*2,out_channels=hidden_units*2,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(hidden_units*2),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )


    self.last_conv=nn.Conv2d(in_channels=hidden_units*2,out_channels=output_shape,kernel_size=1)
    self.pool=nn.AdaptiveAvgPool2d((7,7))

    self.classifier = nn.Sequential(
          nn.Flatten(),
          nn.Linear(in_features=output_shape*7*7, out_features=output_shape)
    )
  def forward(self,x):
    x=self.block1(x)
    x=self.block2(x)
    heatmap=self.last_conv(x)
    x=self.pool(heatmap)
    x=self.classifier(x)
    return x,heatmap

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        return self.block(x)
class Model_v2(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()

        # Backbone
        self.features = nn.Sequential(
            ConvBlock(in_channels, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
            ConvBlock(256, 512)
        )

        # CAM Head
        self.cam_conv = nn.Conv2d(512, num_classes, kernel_size=1)

        # Classifier
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.features(x)           # [B, 512, H, W]
        heatmap = self.cam_conv(x)     # [B, C, H, W]
        logits = self.pool(heatmap)    # [B, C, 1, 1]
        logits = logits.view(x.size(0), -1)

        return logits, heatmap

In [ ]:
model0=Simple_model(input_shape=1,hidden_units=64,output_shape=len(train_data.classes))

In [ ]:
len(train_data.classes)

2

In [ ]:


# print(f"the epoch is {epoch+1} and the loss of it is {loss} also the and the correct choices are {acc/len(train_data_loader)}")
device='cuda' if torch.cuda.is_available() else 'cpu'
# model0.to(device)
modelv1=Modelv1(input_shape=1,hidden_units=64,output_shape=len(train_data.classes))
modelv1.to(device)

Modelv1(
  (block1): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Dropout(p=0.2, inplace=False)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block2): Sequential(
    (0): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Dropout(p=0.2, inplace=False)
    (7): MaxPool2d(kernel_size=2, stride=2, paddin

In [ ]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(params=modelv1.parameters(),lr=0.03)

In [ ]:
from tqdm import tqdm

def train_test(model, num_epochs, patience=3,train_data_loader=train_data_loader,test_data_loader=test_data_loader):

    best_loss = float("inf")
    counter = 0

    for epoch in range(num_epochs):

        # =======================
        # Training
        # =======================
        model.train()
        train_loss, train_acc = 0.0, 0.0

        loop = tqdm(
            train_data_loader,
            desc=f"Epoch [{epoch+1}/{num_epochs}] Training",
            leave=False
        )

        for x, y in loop:
            x,y=x.to(device),y.to(device)
            y_pred, _ = model(x)

            loss = loss_fn(y_pred, y)
            pred = torch.argmax(y_pred, dim=1)
            acc = (pred == y).float().mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_acc += acc.item()

            loop.set_postfix(loss=loss.item(), acc=acc.item())

        train_loss /= len(train_data_loader)
        train_acc /= len(train_data_loader)

        # =======================
        # Testing
        # =======================
        model.eval()
        test_loss, test_acc = 0.0, 0.0

        with torch.inference_mode():
            for x_test, y_test in test_data_loader:
                x_test,y_test=x_test.to(device),y_test.to(device)
                y_pred_test, _ = model(x_test)

                loss_test = loss_fn(y_pred_test, y_test)
                pred_test = torch.argmax(y_pred_test, dim=1)
                acc = (pred_test == y_test).float().mean()

                test_loss += loss_test.item()
                test_acc += acc.item()

        test_loss /= len(test_data_loader)
        test_acc /= len(test_data_loader)

        # =======================
        # Early Stopping Logic
        # =======================
        if test_loss < best_loss:
            best_loss = test_loss
            counter = 0
            best_model_state = model.state_dict()
        else:
            counter += 1

        print(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} || "
            f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} "
            f"| ES Counter: {counter}/{patience}"
        )

        if counter >= patience:
            print("Early stopping triggered.")
            model.load_state_dict(best_model_state)
            break



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelv1=modelv1.to(device)
train_test(modelv1,60,5)

Epoch [1/60] Training:   0%|          | 0/131 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
modelv2=Model_v2(in_channels=1).to(device)
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(params=modelv2.parameters())
train_test(modelv2,60,20)

Epoch [1/60] Train Loss: 0.6293 | Train Acc: 0.6535 || Test Loss: 0.5324 | Test Acc: 0.7309 | ES Counter: 0/10


Epoch [2/60] Train Loss: 0.5843 | Train Acc: 0.6762 || Test Loss: 0.5887 | Test Acc: 0.6944 | ES Counter: 1/10


Epoch [3/60] Train Loss: 0.5462 | Train Acc: 0.7198 || Test Loss: 0.5301 | Test Acc: 0.7083 | ES Counter: 0/10


Epoch [4/60] Train Loss: 0.5257 | Train Acc: 0.7275 || Test Loss: 0.4194 | Test Acc: 0.8038 | ES Counter: 0/10


Epoch [5/60] Train Loss: 0.5255 | Train Acc: 0.7207 || Test Loss: 0.6045 | Test Acc: 0.7101 | ES Counter: 1/10


...

Epoch [33/60] Train Loss: 0.2161 | Train Acc: 0.9140 || Test Loss: 0.2509 | Test Acc: 0.9010 | ES Counter: 6/10


Epoch [34/60] Train Loss: 0.2000 | Train Acc: 0.9174 || Test Loss: 0.3073 | Test Acc: 0.8785 | ES Counter: 7/10


Epoch [35/60] Train Loss: 0.2070 | Train Acc: 0.9151 || Test Loss: 0.2894 | Test Acc: 0.8819 | ES Counter: 8/10


Epoch [36/60] Train Loss: 0.1902 | Train Acc: 0.9263 || Test Loss: 0.4456 | Test Acc: 0.8403 | ES Counter: 9/10


Epoch [37/60] Train Loss: 0.1815 | Train Acc: 0.9277 || Test Loss: 0.4188 | Test Acc: 0.8542 | ES Counter: 10/10
Early stopping triggered.


In [ ]:
        # if test_loss < best_loss:
        #     best_loss = test_loss
        #     counter = 0
        #     best_model_state = model.state_dict()
        # else:
        #     counter += 1

        # print(
        #     f"Epoch [{epoch+1}/{num_epochs}] "
        #     f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} || "
        #     f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} "
        #     f"| ES Counter: {counter}/{patience}"
        # )

        # if counter >= patience:
        #     print("Early stopping triggered.")
        #     model.load_state_dict(best_model_state)
        #     break


In [ ]:
print(type(train_data_loader))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [ ]:
loaded_model=Model_v2(in_channels=1)
model_path="modelv2_weights.pth"
torch.save(modelv2.state_dict(),model_path)
loaded_model.load_state_dict(torch.load(model_path))#first import the weights as file then load it

encoder_state_dict = modelv2.features.state_dict()
torch.save(modelv2.features.state_dict(), "encoder_ct.pth")
loaded_model.eval()

In [ ]:
import os
print(os.getcwd())

/content


#let us now build the second data_model

In [ ]:
from torchvision import transforms,datasets
import torch
from torch.utils.data import DataLoader

In [ ]:
train_transform_mri=transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224,224)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(0.3),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x - x.mean()) / (x.std() + 1e-5))]
)

test_transform_mri=transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.transforms.Lambda(lambda x:(x-x.mean()) / (x.std()+ 1e-5))
])

In [ ]:
train_path_mr="/content/drive/MyDrive/ActualData/train_folder"
test_path_mr="/content/drive/MyDrive/ActualData/test_folder"

In [ ]:
train_data_mr=datasets.ImageFolder(root=train_path_mr,
                                   transform=train_transform_mri,
                                   target_transform=None)
test_data_mr=datasets.ImageFolder(root=test_path_mr,
                                  transform=test_transform_mri,
                                  target_transform=None)

In [ ]:
train_data_mr_loader=DataLoader(dataset=train_data_mr,
                                batch_size=32,
                                shuffle=True)
test_data_mr_loader=DataLoader(dataset=test_data_mr,
                               batch_size=32,
                               shuffle=False)

In [ ]:
images, labels = next(iter(train_data_mr_loader))

print("Images shape:", images.shape)
print("Images dtype:", images.dtype)
print("Labels shape:", labels.shape)
print("Labels dtype:", labels.dtype)

Images shape: torch.Size([32, 1, 224, 224])
Images dtype: torch.float32
Labels shape: torch.Size([32])
Labels dtype: torch.int64


In [ ]:
modelv2_mr=Model_v2(in_channels=1,num_classes=2).to(device)
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(params=modelv2_mr.parameters(),lr=0.03)
train_test(modelv2_mr,60,train_data_loader=train_data_mr_loader,test_data_loader=test_data_mr_loader)


In [ ]:
loaded_model_mr=Model_v2(in_channels=1)
model_path="modelv1_mr_weights.pth"
torch.save(modelv1_mr.state_dict(),model_path )
loaded_model_mr.load_state_dict(torch.load(model_path))
loaded_model_mr.eval()


torch.save(modelv2.features.state_dict(), "encoder_mri.pth")

# encoder_state_dict = torch.load("encoder_mri.pth")